# 30 Days of ML competition_V5_New Paramaters

<font color='blue'>**Version Characteristics:**<font color='black'>
* **Predictive Model: <font color='red'>LGBMRegressor**.<font color='black'>
* **Predictive Model Paramater: <font color='red'>(Many - Please check model code blocl)**.<font color='black'>
* **Evaluation Metric: <font color='red'>RMSE**.<font color='black'>
* **Predictor Variables: <font color='red'>Categorical & Numerical**.<font color='black'>
* **Categorical Variables Handling: <font color='red'>OneHotEncoder() for <= 16**.<font color='black'>
* **Categorical Variables Outlier: <font color='red'>N/A**.<font color='black'>
* **Categorical Variables Handling: <font color='red'>OrdinalEncoder()**.<font color='black'>

Welcome to the **[30 Days of ML competition](https://www.kaggle.com/c/30-days-of-ml/overview)**!  In this notebook, you'll learn how to make your first submission.

Before getting started, make your own editable copy of this notebook by clicking on the **Copy and Edit** button.

# Step 1: Import helpful libraries

We begin by importing the libraries we'll need.  Some of them will be familiar from the **[Intro to Machine Learning](https://www.kaggle.com/learn/intro-to-machine-learning)** course and the **[Intermediate Machine Learning](https://www.kaggle.com/learn/intermediate-machine-learning)** course.

In [78]:
# Main Data Science and Coding Packages
####################################

# Import main data science libraries for dealing with tabular files, datetime, math, and os
import os                             # Handle Operating System directory and commands. May need it, may not
import math as mt                     # Math operations. May need it, may not
import datetime                       # Handle datetime operations. May need it, may not
import numpy as np                    # linear algebra
import pandas as pd                   # data processing, CSV file I/O (e.g. pd.read_csv)

import matplotlib.pyplot as plt       # data visualization package and use call magic (%matplotlib inline) key words matplotlib
%matplotlib inline

import seaborn as sns                 # data visualization package
sns.set_style('darkgrid')             # Set seaboarn default style

import warnings                       # A context manager that copies and restores the warnings filter upon exiting the context
warnings.filterwarnings('ignore')     # Ignore any deprecation warnings that may rise.




# Data Wrangling and Preprocessing
####################################

from sklearn.model_selection import train_test_split      # For splitting data
from sklearn.model_selection import cross_val_score       # For small datasets cross validation 
from sklearn.model_selection import KFold                 # Provides train/test indices to split data in train/test sets.
from sklearn.model_selection import GridSearchCV          # Exhaustive search over specified parameter values for an estimator

from sklearn.impute import SimpleImputer                  # For data imputation

from sklearn.preprocessing import OrdinalEncoder          # For ordinal encoding categorical variables 
from sklearn.preprocessing import OneHotEncoder           # For OneHot encoding categorical variables 
from sklearn.preprocessing import LabelEncoder            # For y transformation, not for X - also for non-numerical labels

from sklearn.pipeline import Pipeline                     # For Pipeline setting 
from sklearn.compose import ColumnTransformer             # For Pipeline setting

import optuna                                             # A hyperparameter optimization framework
from functools import partial                             # new function with partial application of the given arguments 
                                                          # and keywords

# ML Models Packages
####################################

# For training linear regression model
from sklearn.linear_model import LinearRegression

# For training random forest model
from sklearn.ensemble import RandomForestRegressor

# For training decision tree model
from sklearn.tree import DecisionTreeRegressor

# For training extreme gradient boosting model
from xgboost import XGBRegressor


# For training LGBMRegressor model
from lightgbm import LGBMRegressor         # For Build a gradient boosting model from the training set (X, y)
from tqdm import tqdm                      # Decorate an iterable object, returning an iterator which acts exactly 
                                           # | like the original iterable, but prints a dynamically updating
                                           # |  progressbar every time a value is requested.



# ML Models Evaluation Metrices
##############################

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import max_error
from sklearn.metrics import r2_score
from sklearn.metrics import accuracy_score

print("All needed packages successfully imported")

All needed packages successfully imported


# Step 2: Load the data

Next, we'll load the training and test data.  

We set `index_col=0` in the code cell below to use the `id` column to index the DataFrame.  (*If you're not sure how this works, try temporarily removing `index_col=0` and see how it changes the result.*)

In [79]:
# Load the training data
train = pd.read_csv("../input/30-days-of-ml/train.csv", index_col=0)
test = pd.read_csv("../input/30-days-of-ml/test.csv", index_col=0)

In [80]:
# Preview the training data
print("A preview of training data:")
train.head()

A preview of training data:


,cat0,cat1,cat2,cat3,cat4,cat5,cat6,cat7,cat8,cat9,...,cont5,cont6,cont7,cont8,cont9,cont10,cont11,cont12,cont13,target
id,,,,,,,,,,,,,,,,,,,,,
1,B,B,B,C,B,B,A,E,C,N,...,0.400361,0.160266,0.310921,0.389470,0.267559,0.237281,0.377873,0.322401,0.869850,8.113634
2,B,B,A,A,B,D,A,F,A,O,...,0.533087,0.558922,0.516294,0.594928,0.341439,0.906013,0.921701,0.261975,0.465083,8.481233
3,A,A,A,C,B,D,A,D,A,F,...,0.650609,0.375348,0.902567,0.555205,0.843531,0.748809,0.620126,0.541474,0.763846,8.364351
4,B,B,A,C,B,D,A,E,C,K,...,0.668980,0.239061,0.732948,0.679618,0.574844,0.346010,0.714610,0.540150,0.280682,8.049253
6,A,A,A,C,B,D,A,E,A,N,...,0.686964,0.420667,0.648182,0.684501,0.956692,1.000773,0.776742,0.625849,0.250823,7.972260


In [81]:
# Preview the test data
print("A preview of test data:")
test.head()

A preview of test data:


,cat0,cat1,cat2,cat3,cat4,cat5,cat6,cat7,cat8,cat9,...,cont4,cont5,cont6,cont7,cont8,cont9,cont10,cont11,cont12,cont13
id,,,,,,,,,,,,,,,,,,,,,
0,B,B,B,C,B,B,A,E,E,I,...,0.476739,0.376350,0.337884,0.321832,0.445212,0.290258,0.244476,0.087914,0.301831,0.845702
5,A,B,A,C,B,C,A,E,C,H,...,0.285509,0.860046,0.798712,0.835961,0.391657,0.288276,0.549568,0.905097,0.850684,0.693940
15,B,A,A,A,B,B,A,E,D,K,...,0.697272,0.683600,0.404089,0.879379,0.275549,0.427871,0.491667,0.384315,0.376689,0.508099
16,B,B,A,C,B,D,A,E,A,N,...,0.719306,0.777890,0.730954,0.644315,1.024017,0.391090,0.988340,0.411828,0.393585,0.461372
17,B,B,A,C,B,C,A,E,C,F,...,0.313032,0.431007,0.390992,0.408874,0.447887,0.390253,0.648932,0.385935,0.370401,0.900412


In [82]:
# # Print summary statistics for training data
# print("Summary statistics for training data:\n", train.describe())

# print("-" * 60)

# # Print summary statistics for test data
# print("\nSummary statistics for test data:\n", test.describe())

In [83]:
# Print column names for training data
print("Training data columns name:\n", train.columns)

print("----------" * 10)

# Print column names for test data
print("Test data columns name:\n", test.columns)

Training data columns name:
 Index(['cat0', 'cat1', 'cat2', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8',
       'cat9', 'cont0', 'cont1', 'cont2', 'cont3', 'cont4', 'cont5', 'cont6',
       'cont7', 'cont8', 'cont9', 'cont10', 'cont11', 'cont12', 'cont13',
       'target'],
      dtype='object')
----------------------------------------------------------------------------------------------------
Test data columns name:
 Index(['cat0', 'cat1', 'cat2', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8',
       'cat9', 'cont0', 'cont1', 'cont2', 'cont3', 'cont4', 'cont5', 'cont6',
       'cont7', 'cont8', 'cont9', 'cont10', 'cont11', 'cont12', 'cont13'],
      dtype='object')


In [84]:
# Print Shape of training data (num_rows, num_columns)
print("Training data shape:\n",train.shape)

print("-" * 60)

# Print Shape of test data (num_rows, num_columns)
print("Test data shape:\n", test.shape)

Training data shape:
 (300000, 25)
------------------------------------------------------------
Test data shape:
 (200000, 24)


In [85]:
# Number of missing values in each column of training data
missing_val_count_by_column_train = (train.isnull().sum())
print("Number of missing values in each column of training data")
print(missing_val_count_by_column_train[missing_val_count_by_column_train > 0])

print("-" * 60)

# Number of missing values in each column of test data
missing_val_count_by_column_test = (test.isnull().sum())
print("Number of missing values in each column of test data")
print(missing_val_count_by_column_test[missing_val_count_by_column_test > 0])

Number of missing values in each column of training data
Series([], dtype: int64)
------------------------------------------------------------
Number of missing values in each column of test data
Series([], dtype: int64)


**Make 3 lists for train dataset columns (All, numerical, and categorical)**

In [86]:
#  X columns (All, numerical, and categorical)

# List of numerical columns
num_cols = [cname for cname in train.columns if train[cname].dtype in ['int64', 'float64']]

# List of categorical columns
object_cols = [col for col in train.columns if train[col].dtype == "object"]

# List of all columns = 
all_cols =  object_cols + num_cols


print('Numerical columns are:\n', num_cols)
print('-' * 60)
print('Categorical columns are:\n', object_cols)
print('-' * 60)
print('All columns are:\n', all_cols)
print('-' * 60)

Numerical columns are:
 ['cont0', 'cont1', 'cont2', 'cont3', 'cont4', 'cont5', 'cont6', 'cont7', 'cont8', 'cont9', 'cont10', 'cont11', 'cont12', 'cont13', 'target']
------------------------------------------------------------
Categorical columns are:
 ['cat0', 'cat1', 'cat2', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8', 'cat9']
------------------------------------------------------------
All columns are:
 ['cat0', 'cat1', 'cat2', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8', 'cat9', 'cont0', 'cont1', 'cont2', 'cont3', 'cont4', 'cont5', 'cont6', 'cont7', 'cont8', 'cont9', 'cont10', 'cont11', 'cont12', 'cont13', 'target']
------------------------------------------------------------


* **Plotting train dataset using seaboarn scatterplot for more data understandin**

In [87]:
# # Make scatter plot for each numerical predictor variable versus target
# for col in train[num_cols]:
#     plt.figure()
#     sns.scatterplot(x=train[col], y=y, data=train)

In [88]:
# # Make CAT scatter plot for each categorical predictor variable versus target
# for col in train[object_cols]:
#     plt.figure()
#     sns.catplot(x=train[col], y=y, jitter=False, data=train)

* **DROP columns with Low correlation with target variable**
    * Based on above plots, we cause assume that below col has low correlation with target variable, and;
    * It's better to be dropped from our train data.

In [89]:
num_cols_to_remove_list = ['cont1', 'cont8']
object_cols_to_remove_list = ['cat0', 'cat1', 'cat2', 'cat9']

for num_element in num_cols:
    if num_element in num_cols_to_remove_list:
        num_cols.remove(num_element)

for object_element in object_cols:
    if object_element in object_cols_to_remove_list:
        object_cols.remove(object_element)

# List of all columns after dropping columns with low correlation 
all_cols =  object_cols + num_cols


print('Final Numerical columns are:\n', num_cols)
print('-' * 60)
print('Final Categorical columns are:\n', object_cols)
print('-' * 60)
print('Final All columns are:\n', all_cols)
print('-' * 60)

Final Numerical columns are:
 ['cont0', 'cont2', 'cont3', 'cont4', 'cont5', 'cont6', 'cont7', 'cont9', 'cont10', 'cont11', 'cont12', 'cont13', 'target']
------------------------------------------------------------
Final Categorical columns are:
 ['cat1', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8']
------------------------------------------------------------
Final All columns are:
 ['cat1', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8', 'cont0', 'cont2', 'cont3', 'cont4', 'cont5', 'cont6', 'cont7', 'cont9', 'cont10', 'cont11', 'cont12', 'cont13', 'target']
------------------------------------------------------------


In [90]:
# Adjut train dataset to has only our selected columns
train = train[all_cols]

In [91]:
# Preview the training data
print("A preview of training data:")
train.head()

A preview of training data:


,cat1,cat3,cat4,cat5,cat6,cat7,cat8,cont0,cont2,cont3,cont4,cont5,cont6,cont7,cont9,cont10,cont11,cont12,cont13,target
id,,,,,,,,,,,,,,,,,,,,
1,B,C,B,B,A,E,C,0.201470,0.669699,0.136278,0.610706,0.400361,0.160266,0.310921,0.267559,0.237281,0.377873,0.322401,0.869850,8.113634
2,B,A,B,D,A,F,A,0.743068,1.021605,0.365798,0.276853,0.533087,0.558922,0.516294,0.341439,0.906013,0.921701,0.261975,0.465083,8.481233
3,A,C,B,D,A,D,A,0.742708,-0.012673,0.576957,0.285074,0.650609,0.375348,0.902567,0.843531,0.748809,0.620126,0.541474,0.763846,8.364351
4,B,C,B,D,A,E,C,0.429551,0.577942,0.280610,0.284667,0.668980,0.239061,0.732948,0.574844,0.346010,0.714610,0.540150,0.280682,8.049253
6,A,C,B,D,A,E,A,1.058291,-0.052389,0.232407,0.287595,0.686964,0.420667,0.648182,0.956692,1.000773,0.776742,0.625849,0.250823,7.972260


In [92]:
# Print Shape of training data (num_rows, num_columns)
print("Training data shape:\n",train.shape)

print("-" * 60)

# Print Shape of test data (num_rows, num_columns)
print("Test data shape:\n", test.shape)

Training data shape:
 (300000, 20)
------------------------------------------------------------
Test data shape:
 (200000, 24)


The next code cell separates the target (which we assign to `y`) from the training features (which we assign to `features`).

In [93]:
# Separate target from features

# Assign target to y variable [pandas Series]
y = train['target']

# Assign all predictor variables to feature variable (by dropping target y) 
features = train.drop(['target'], axis=1)
# Preview features
features.head()

# Assign predictor variables to X and X_test variables [pandas DFs]
X = features.copy()
X_test = test.copy()

# Step 3: Prepare the data

Next, we'll need to handle the categorical columns (`cat0`, `cat1`, ... `cat9`).  

In the **[Categorical Variables lesson](https://www.kaggle.com/alexisbcook/categorical-variables)** in the Intermediate Machine Learning course, you learned several different ways to encode categorical variables in a dataset.  In this notebook, we'll use **one-hot encoding** and save our encoded features as new variables `X` and `X_test`.

**Get X columns (All, numerical, and categorical)**

In [94]:
#  X columns (All, numerical, and categorical)

# List of numerical columns
num_cols = [cname for cname in X.columns if X[cname].dtype in ['int64', 'float64']]

# List of categorical columns
object_cols = [col for col in X.columns if X[col].dtype == "object"]

# List of all columns = 
all_cols =  object_cols + num_cols


print('Numerical columns are:\n', num_cols)
print('-' * 60)
print('Categorical columns are:\n', object_cols)
print('-' * 60)
print('All columns are:\n', all_cols)
print('-' * 60)

Numerical columns are:
 ['cont0', 'cont2', 'cont3', 'cont4', 'cont5', 'cont6', 'cont7', 'cont9', 'cont10', 'cont11', 'cont12', 'cont13']
------------------------------------------------------------
Categorical columns are:
 ['cat1', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8']
------------------------------------------------------------
All columns are:
 ['cat1', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8', 'cont0', 'cont2', 'cont3', 'cont4', 'cont5', 'cont6', 'cont7', 'cont9', 'cont10', 'cont11', 'cont12', 'cont13']
------------------------------------------------------------


In [95]:
# Adjut X dataset to has only our selected columns
X = X[all_cols]
X.columns

Index(['cat1', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8', 'cont0',
       'cont2', 'cont3', 'cont4', 'cont5', 'cont6', 'cont7', 'cont9', 'cont10',
       'cont11', 'cont12', 'cont13'],
      dtype='object')

**Get Safe columns (Columns that exist at "test dataset")**

In [96]:
# Safe Columns

# Columns that can be safely used ((becasue it exist at "test dataset"))
good_label_cols = [col for col in X.columns if col in X_test.columns]

        
# Problematic columns that will be dropped from the dataset ((becasue it does NOT exist at "test dataset"))
bad_label_cols = list(set(all_cols)-set(good_label_cols))

print('Columns that can be safely used (EXIST at test dataset):\n', good_label_cols)
print("-" * 60)
print('Columns that CANNOT be safely used (Does NOT exist at test dataset):\n', bad_label_cols)
print("-" * 60)


# Assign "Safe Columns" to X dataframe
X = X[good_label_cols]
print("-" * 60)
print('X Safe Columns are:\n', X.columns)

Columns that can be safely used (EXIST at test dataset):
 ['cat1', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8', 'cont0', 'cont2', 'cont3', 'cont4', 'cont5', 'cont6', 'cont7', 'cont9', 'cont10', 'cont11', 'cont12', 'cont13']
------------------------------------------------------------
Columns that CANNOT be safely used (Does NOT exist at test dataset):
 []
------------------------------------------------------------
------------------------------------------------------------
X Safe Columns are:
 Index(['cat1', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8', 'cont0',
       'cont2', 'cont3', 'cont4', 'cont5', 'cont6', 'cont7', 'cont9', 'cont10',
       'cont11', 'cont12', 'cont13'],
      dtype='object')


In [97]:
X.columns

Index(['cat1', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8', 'cont0',
       'cont2', 'cont3', 'cont4', 'cont5', 'cont6', 'cont7', 'cont9', 'cont10',
       'cont11', 'cont12', 'cont13'],
      dtype='object')

In [98]:
X_test.columns

Index(['cat0', 'cat1', 'cat2', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8',
       'cat9', 'cont0', 'cont1', 'cont2', 'cont3', 'cont4', 'cont5', 'cont6',
       'cont7', 'cont8', 'cont9', 'cont10', 'cont11', 'cont12', 'cont13'],
      dtype='object')

**Get Categorical columns "Cardinality" ("Cardinality" means the number of unique values in a column)**

In [99]:
# Get number of unique entries in each column with categorical data
object_nunique = list(map(lambda col: X[col].nunique(), object_cols))
d = dict(zip(object_cols, object_nunique))

# Print number of unique entries by column, in ascending order
print('Categorical columns Cardinality:\n', sorted(d.items(), key=lambda x: x[1]))
print("-" * 60)
# Columns that will be one-hot encoded
low_cardinality_cols = [col for col in object_cols if X[col].nunique() <= 16]

# Columns that will be dropped from the dataset
high_cardinality_cols = list(set(object_cols)-set(low_cardinality_cols))

print('Categorical columns that will be encoded:', low_cardinality_cols)
print('\nCategorical columns that will be dropped from the dataset:', high_cardinality_cols)
print("-" * 60)


Categorical columns Cardinality:
 [('cat1', 2), ('cat3', 4), ('cat4', 4), ('cat5', 4), ('cat8', 7), ('cat6', 8), ('cat7', 8)]
------------------------------------------------------------
Categorical columns that will be encoded: ['cat1', 'cat3', 'cat4', 'cat5', 'cat6', 'cat7', 'cat8']

Categorical columns that will be dropped from the dataset: []
------------------------------------------------------------


* **Apply encoding method**

In [100]:
# Set test data to has the same columns for validation purpose during model training phase
X_test = X_test[all_cols]

label = LabelEncoder()

for column in low_cardinality_cols:
        label.fit(X[column])
        X[column] = label.transform(X[column])
        X_test[column] = label.transform(X_test[column])






**Data Splitting**

Next, we break off a validation set from the training data.

In [101]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)

In [102]:
#Check final data shap

# Print Shape of X_train data (num_rows, num_columns)
print("X_train shape:\n", X_train.shape)

print("-" * 60)

# Print Shape of y_train data (num_rows, num_columns)
print("y_train:\n", y_train.shape)

print("-" * 60)

# Print Shape of X_valid data (num_rows, num_columns)
print("X_valid shape:\n", X_valid.shape)

print("-" * 60)

# Print Shape of y_valid data (num_rows, num_columns)
print("y_valid shape:\n", y_valid.shape)

print("-" * 60)
print("-" * 60)

# Print Shape of test data (num_rows, num_columns)
print("test data shape:\n", X_test.shape)

X_train shape:
 (240000, 19)
------------------------------------------------------------
y_train:
 (240000,)
------------------------------------------------------------
X_valid shape:
 (60000, 19)
------------------------------------------------------------
y_valid shape:
 (60000,)
------------------------------------------------------------
------------------------------------------------------------
test data shape:
 (200000, 19)


# Step 4: Train a model

Now that the data is prepared, the next step is to train a model.  

If you took the **[Intro to Machine Learning](https://www.kaggle.com/learn/intro-to-machine-learning)** courses, then you learned about **[Random Forests](https://www.kaggle.com/dansbecker/random-forests)**.  In the code cell below, we fit a random forest model to the data.

In [105]:
# Define the model 

# Set lgbm_parameters
        
lgbm_parameters = {'max_depth': 43, 
                   'subsample': 0.5203056390091917, 
                   'colsample_bytree': 0.20060402516346498, 
                   'learning_rate': 0.009545917197979484, 
                   'reg_lambda': 21.855676691734885, 
                   'reg_alpha': 15.43291267463823, 
                   'min_child_samples': 24, 
                   'num_leaves': 10, 
                   'max_bin': 839, 
                   'cat_smooth': 86,
                   'metric': 'rmse',
                   'n_jobs': -1,
                   'n_estimators': 2000,
                   'cat_l2': 11.3562583709454,
                  }


# Define, Fit, and Evaluate LGBMRegressor Model


lgbm_val_pred = np.zeros(len(y))
lgbm_test_pred = np.zeros(len(X_test))
rmse = []
kf = KFold(n_splits=2, shuffle=True)

for trn_idx, val_idx in tqdm(kf.split(X,y)):
    x_train_idx = X.iloc[trn_idx]
    y_train_idx = y.iloc[trn_idx]
    x_valid_idx = X.iloc[val_idx]
    y_valid_idx = y.iloc[val_idx]
    
    # Train the model (will take about 10 minutes to run)
    lgbm_model = LGBMRegressor(**lgbm_parameters)
    
    # Train the model (will take about 10 minutes to run)
    lgbm_model.fit(x_train_idx, y_train_idx, eval_set = ((x_valid_idx,y_valid_idx)),verbose = -1, early_stopping_rounds = 50,categorical_feature=low_cardinality_cols)
    lgbm_test_pred += lgbm_model.predict(X_test)/2
    rmse.append(mean_squared_error(y_valid_idx, lgbm_model.predict(x_valid_idx), squared=False))

    
# Geting and print the Average of RMSE
print(min(rmse))
print(max(rmse))

rmse = sum(rmse)/len(rmse)
print(rmse)

0it [00:00, ?it/s]

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's rmse: 0.724941


1it [00:33, 33.38s/it]

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's rmse: 0.726063


2it [01:06, 33.40s/it]

0.7249405689014746
0.7260630821457458
0.7255018255236102


In the code cell above, we set `squared=False` to get the root mean squared error (RMSE) on the validation data.

# Step 5: Submit to the competition

We'll begin by using the trained model to generate predictions, which we'll save to a CSV file.

In [106]:
# Use the model to generate predictions
predictions = lgbm_model.predict(X_test)

# Save the predictions to a CSV file
output = pd.DataFrame({'Id': X_test.index,
                       'target': predictions})
output.to_csv('submission.csv', index=False)

Once you have run the code cell above, follow the instructions below to submit to the competition:
1. Begin by clicking on the **Save Version** button in the top right corner of the window.  This will generate a pop-up window.  
2. Ensure that the **Save and Run All** option is selected, and then click on the **Save** button.
3. This generates a window in the bottom left corner of the notebook.  After it has finished running, click on the number to the right of the **Save Version** button.  This pulls up a list of versions on the right of the screen.  Click on the ellipsis **(...)** to the right of the most recent version, and select **Open in Viewer**.  This brings you into view mode of the same page. You will need to scroll down to get back to these instructions.
4. Click on the **Output** tab on the right of the screen.  Then, click on the file you would like to submit, and click on the **Submit** button to submit your results to the leaderboard.

You have now successfully submitted to the competition!

If you want to keep working to improve your performance, select the **Edit** button in the top right of the screen. Then you can change your code and repeat the process. There's a lot of room to improve, and you will climb up the leaderboard as you work.

# Step 6: Keep Learning!

If you're not sure what to do next, you can begin by trying out more model types!
1. If you took the **[Intermediate Machine Learning](https://www.kaggle.com/learn/intermediate-machine-learning)** course, then you learned about **[XGBoost](https://www.kaggle.com/alexisbcook/xgboost)**.  Try training a model with XGBoost, to improve over the performance you got here.

2. Take the time to learn about **Light GBM (LGBM)**, which is similar to XGBoost, since they both use gradient boosting to iteratively add decision trees to an ensemble.  In case you're not sure how to get started, **[here's a notebook](https://www.kaggle.com/svyatoslavsokolov/tps-feb-2021-lgbm-simple-version)** that trains a model on a similar dataset.